In [1]:
# Cell 1 - Clone EasyEdit fresh
%cd /kaggle/working
!rm -rf EasyEdit
!git clone https://github.com/zjunlp/EasyEdit.git
%cd /kaggle/working/EasyEdit

/kaggle/working
Cloning into 'EasyEdit'...
remote: Enumerating objects: 10275, done.
remote: Counting objects: 100% (2007/2007), done.
remote: Compressing objects: 100% (558/558), done.
remote: Total 10275 (delta 1611), reused 1449 (delta 1449), pack-reused 8268 (from 2)
Receiving objects: 100% (10275/10275), 96.42 MiB | 42.98 MiB/s, done.
Resolving deltas: 100% (6614/6614), done.
/kaggle/working/EasyEdit


In [2]:
# Cell 2 - Create pydeps directory
!rm -rf /kaggle/working/pydeps
!mkdir -p /kaggle/working/pydeps

In [3]:
# Cell 3 - Install dependencies
!python -m pip install --no-cache-dir --upgrade --target /kaggle/working/pydeps \
  "numpy==1.26.4" \
  "scipy==1.13.1" \
  "scikit-learn==1.5.2" \
  "PyYAML==6.0.2" \
  "transformers==4.45.2" \
  "tokenizers==0.20.3" \
  "sentence-transformers==3.2.1" \
  "accelerate>=0.30.0" \
  "datasets" \
  "einops" \
  "higher" \
  "hydra-core" \
  "omegaconf" \
  "peft" \
  "tqdm" \
  "nltk" \
  "pandas" \
  "rouge==1.0.1" \
  "openai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 269.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 275.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 297.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 264.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 253.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 163.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 338.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 238.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 303.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 242.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 130.2

In [4]:
# Cell 4 - Fix sys.path: pydeps first, dist-packages last
import sys

PYDEPS = "/kaggle/working/pydeps"
REPO = "/kaggle/working/EasyEdit"

# Strip dist-packages first to prevent Kaggle/global packages from loading first
sys.path = [p for p in sys.path if 'dist-packages' not in p]

sys.path = [
    PYDEPS,
    REPO,
    "/usr/lib/python3.12",
    "/usr/lib/python3.12/lib-dynload",
    "/usr/local/lib/python3.12/dist-packages",  # needed for kaggle_secrets, torch, etc.
]

# Important: if the kernel imported old packages before this cell, remove them from memory.
# In a clean restart this does nothing, but it prevents accidentally using Kaggle's numpy/transformers.
for name in list(sys.modules):
    if name == "numpy" or name.startswith("numpy.")        or name == "scipy" or name.startswith("scipy.")        or name == "sklearn" or name.startswith("sklearn.")        or name == "transformers" or name.startswith("transformers.")        or name == "tokenizers" or name.startswith("tokenizers."):
        del sys.modules[name]

print(sys.path)

import transformers
print("transformers:", transformers.__version__, transformers.__file__)
# Must show 4.45.2 from /kaggle/working/pydeps


['/kaggle/working/pydeps', '/kaggle/working/EasyEdit', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '/usr/local/lib/python3.12/dist-packages']
transformers: 4.45.2 /kaggle/working/pydeps/transformers/__init__.py


In [5]:
# Patch LlamaModel.forward layer_outputs safely
from pathlib import Path
import py_compile

llama_path = Path("/kaggle/working/pydeps/transformers/models/llama/modeling_llama.py")
text = llama_path.read_text()

old = "            hidden_states = layer_outputs[0]"

new = '''            # ROME/EasyEdit compatibility: support tuple/list/dict layer outputs
            if isinstance(layer_outputs, dict):
                def _first_tensor_from_layer_output(obj):
                    import torch
                    if torch.is_tensor(obj):
                        return obj
                    if isinstance(obj, dict):
                        for key in ["hidden_states", "last_hidden_state"]:
                            if key in obj:
                                found = _first_tensor_from_layer_output(obj[key])
                                if found is not None:
                                    return found
                        for value in obj.values():
                            found = _first_tensor_from_layer_output(value)
                            if found is not None:
                                return found
                        return None
                    if isinstance(obj, (tuple, list)):
                        for value in obj:
                            found = _first_tensor_from_layer_output(value)
                            if found is not None:
                                return found
                        return None
                    return None

                hidden_states = _first_tensor_from_layer_output(layer_outputs)
                if hidden_states is None:
                    raise TypeError(f"Could not extract tensor from layer_outputs: {layer_outputs}")
            else:
                hidden_states = layer_outputs[0]'''

if old in text and "_first_tensor_from_layer_output" not in text:
    text = text.replace(old, new)
    llama_path.write_text(text)
    print("✅ Applied safer layer_outputs patch")
elif "_first_tensor_from_layer_output" in text:
    print("✅ Safer layer_outputs patch already applied")
else:
    print("⚠️ Original layer_outputs line not found. Maybe already patched.")

py_compile.compile(str(llama_path), doraise=True)
print("✅ modeling_llama.py syntax OK")

✅ Applied safer layer_outputs patch
✅ modeling_llama.py syntax OK


In [6]:
# Cell 5 - Verify all imports
import numpy, scipy, sklearn, transformers, torch

print("numpy:", numpy.__version__, numpy.__file__)
print("scipy:", scipy.__version__, scipy.__file__)
print("sklearn:", sklearn.__version__, sklearn.__file__)
print("transformers:", transformers.__version__, transformers.__file__)
print("torch:", torch.__version__, torch.__file__)
print("cuda:", torch.cuda.is_available())

numpy: 1.26.4 /kaggle/working/pydeps/numpy/__init__.py
scipy: 1.13.1 /kaggle/working/pydeps/scipy/__init__.py
sklearn: 1.5.2 /kaggle/working/pydeps/sklearn/__init__.py
transformers: 4.45.2 /kaggle/working/pydeps/transformers/__init__.py
torch: 2.11.0+cu130 /kaggle/working/pydeps/torch/__init__.py
cuda: True


In [7]:
# Cell 6 - Patch all EasyEdit __init__.py files
from pathlib import Path
import shutil

ROOT = Path("/kaggle/working/EasyEdit/easyeditor")

patches = {
    ROOT / "__init__.py": """from .dataset import *
from .editors import *
from .util import *
""",
    ROOT / "dataset" / "__init__.py": """from .zsre import ZsreDataset
""",
    ROOT / "editors" / "__init__.py": """from .editor import *
""",
    ROOT / "models" / "__init__.py": """from .ike import *
from .rome import *
""",
    ROOT / "trainer" / "__init__.py": """# Minimal - no multimodal
""",
    ROOT / "evaluate" / "__init__.py": """from .evaluate import *
from .evaluate_utils import *
""",
}

for path, text in patches.items():
    backup = path.with_suffix(path.suffix + ".bak")
    if not backup.exists():
        shutil.copy2(path, backup)
    path.write_text(text)
    print(f"Patched: {path.name}")

print("All __init__.py files patched")

Patched: __init__.py
Patched: __init__.py
Patched: __init__.py
Patched: __init__.py
Patched: __init__.py
Patched: __init__.py
All __init__.py files patched


In [8]:
# Cell 7 - Patch alg_dict.py
from pathlib import Path
import shutil

alg_dict_path = Path("/kaggle/working/EasyEdit/easyeditor/util/alg_dict.py")
backup = alg_dict_path.with_suffix(".py.bak")
if not backup.exists():
    shutil.copy2(alg_dict_path, backup)

alg_dict_path.write_text("""from ..models.ike import IKEHyperParams, apply_ike_to_model
from ..models.rome import ROMEHyperParams, apply_rome_to_model

ALG_DICT = {
    "IKE": apply_ike_to_model,
    "ROME": apply_rome_to_model,
}

ALG_MULTIMODAL_DICT = {}

HPARAMS_DICT = {
    "IKE": IKEHyperParams,
    "ROME": ROMEHyperParams,
}
""")

print("Patched alg_dict.py")
print(alg_dict_path.read_text())

Patched alg_dict.py
from ..models.ike import IKEHyperParams, apply_ike_to_model
from ..models.rome import ROMEHyperParams, apply_rome_to_model

ALG_DICT = {
    "IKE": apply_ike_to_model,
    "ROME": apply_rome_to_model,
}

ALG_MULTIMODAL_DICT = {}

HPARAMS_DICT = {
    "IKE": IKEHyperParams,
    "ROME": ROMEHyperParams,
}



In [9]:
# Cell 9 - Patch evaluate_utils.py: make openai optional
from pathlib import Path

eval_utils_path = Path("/kaggle/working/EasyEdit/easyeditor/evaluate/evaluate_utils.py")
content = eval_utils_path.read_text()

content = content.replace(
    "import openai\nfrom openai import OpenAI",
    """try:\n    import openai\n    from openai import OpenAI\nexcept ImportError:\n    pass"""
)

eval_utils_path.write_text(content)
print("Patched evaluate_utils.py")

Patched evaluate_utils.py


In [10]:
# Cell 8 - Patch editor.py: remove melo + stub compute_sent_metric
from pathlib import Path

editor_path = Path("/kaggle/working/EasyEdit/easyeditor/editors/editor.py")
content = editor_path.read_text()

# Remove melo import
content = content.replace(
    "from ..models.melo.melo import LORA",
    "# from ..models.melo.melo import LORA  # disabled"
)

# Stub compute_sent_metric
content = content.replace(
    "from ..evaluate import compute_edit_quality, compute_icl_edit_quality, compute_sent_metric",
    """from ..evaluate import compute_edit_quality, compute_icl_edit_quality
try:
    from ..evaluate import compute_sent_metric
except ImportError:
    def compute_sent_metric(*args, **kwargs): return {}"""
)

editor_path.write_text(content)
print("Patched editor.py")

Patched editor.py


In [11]:
# Cell 10 - Replace EasyEdit nethook Trace/TraceDict with a safe ROME/LLaMA version
# This avoids the broken forward hook behavior that can replace tensor outputs with dicts.
from pathlib import Path
import py_compile

nethook_path = Path("/kaggle/working/EasyEdit/easyeditor/util/nethook.py")
text = nethook_path.read_text()

marker = "# --- BEGIN ROME_LLAMA_SAFE_NETHOOK_OVERRIDE ---"

safe_override = r'''
# --- BEGIN ROME_LLAMA_SAFE_NETHOOK_OVERRIDE ---
# Safe override for EasyEdit ROME + LLaMA tracing.
# The important rule: forward hooks must return the real module output,
# not the saved/cached output object.

import contextlib
from collections import OrderedDict


def _rome_get_module(root, layer):
    # Resolve a dotted layer path like model.layers.0.mlp.
    if layer is None:
        return root
    if not isinstance(layer, str):
        return layer

    module = root
    for part in layer.split("."):
        if part == "":
            continue
        if part.isdigit():
            module = module[int(part)]
        else:
            module = getattr(module, part)
    return module


def _rome_first_tensor(obj):
    # Return the first tensor inside obj, or None if no tensor exists.
    import torch

    if torch.is_tensor(obj):
        return obj

    if isinstance(obj, dict):
        for key in ["hidden_states", "last_hidden_state", "output", "value"]:
            if key in obj:
                found = _rome_first_tensor(obj[key])
                if found is not None:
                    return found
        for value in obj.values():
            found = _rome_first_tensor(value)
            if found is not None:
                return found
        return None

    if isinstance(obj, (tuple, list)):
        for value in obj:
            found = _rome_first_tensor(value)
            if found is not None:
                return found
        return None

    return None


class Trace(contextlib.AbstractContextManager):
    def __init__(
        self,
        module,
        layer=None,
        retain_output=True,
        retain_input=False,
        clone=False,
        detach=False,
        retain_grad=False,
        edit_output=None,
        stop=False,
    ):
        self.root = module
        self.layer = layer
        self.module = _rome_get_module(module, layer)
        self.retain_output = retain_output
        self.retain_input = retain_input
        self.clone = clone
        self.detach = detach
        self.retain_grad = retain_grad
        self.edit_output = edit_output
        self.stop = stop
        self.input = None
        self.output = None
        self.registered_hook = None

    def __enter__(self):
        def retain_hook(m, inputs, output):
            if self.edit_output is not None:
                if "invoke_with_optional_args" in globals():
                    output = invoke_with_optional_args(
                        self.edit_output,
                        output=output,
                        layer=self.layer,
                        inputs=inputs,
                    )
                else:
                    output = self.edit_output(output)

            if self.retain_input:
                saved_input = inputs[0] if len(inputs) == 1 else inputs
                self.input = recursive_copy(
                    saved_input,
                    clone=self.clone,
                    detach=self.detach,
                    retain_grad=False,
                )

            if self.retain_output:
                saved_output = output
                found_tensor = _rome_first_tensor(saved_output)
                if found_tensor is not None:
                    saved_output = found_tensor

                self.output = recursive_copy(
                    saved_output,
                    clone=self.clone,
                    detach=self.detach,
                    retain_grad=self.retain_grad,
                )

            if self.stop:
                raise StopForward()

            # Critical: return the real model output.
            return output

        self.registered_hook = self.module.register_forward_hook(retain_hook)
        return self

    def close(self):
        if self.registered_hook is not None:
            self.registered_hook.remove()
            self.registered_hook = None

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()
        return exc_type is StopForward


class TraceDict(OrderedDict, contextlib.AbstractContextManager):
    def __init__(
        self,
        module,
        layers=None,
        retain_output=True,
        retain_input=False,
        clone=False,
        detach=False,
        retain_grad=False,
        edit_output=None,
        stop=False,
    ):
        OrderedDict.__init__(self)
        self.module = module
        self.layers = [] if layers is None else list(layers)
        self.retain_output = retain_output
        self.retain_input = retain_input
        self.clone = clone
        self.detach = detach
        self.retain_grad = retain_grad
        self.edit_output = edit_output
        self.stop = stop
        self.traces = []

    def __enter__(self):
        for layer in self.layers:
            trace = Trace(
                self.module,
                layer=layer,
                retain_output=self.retain_output,
                retain_input=self.retain_input,
                clone=self.clone,
                detach=self.detach,
                retain_grad=self.retain_grad,
                edit_output=self.edit_output,
                stop=self.stop,
            )
            self[layer] = trace.__enter__()
            self.traces.append(trace)
        return self

    def close(self):
        for trace in reversed(self.traces):
            trace.close()
        self.traces = []

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()
        return exc_type is StopForward
# --- END ROME_LLAMA_SAFE_NETHOOK_OVERRIDE ---
'''

# Remove any old/broken copy of this override, then append the correct one.
if marker in text:
    text = text.split(marker)[0].rstrip() + "\n\n" + safe_override + "\n"
else:
    text = text.rstrip() + "\n\n" + safe_override + "\n"

nethook_path.write_text(text)
py_compile.compile(str(nethook_path), doraise=True)
print("✅ Safe nethook override installed")
print("✅ nethook.py syntax OK")

✅ Safe nethook override installed
✅ nethook.py syntax OK


In [12]:
# Cell 12 - Uninstall torchvision (causes CUDA version conflict)
!python -m pip uninstall -y torchvision

import shutil
from pathlib import Path
for p in Path("/usr/local/lib/python3.12/dist-packages").glob("torchvision*"):
    shutil.rmtree(p, ignore_errors=True)
    print(f"Removed: {p}")
print("torchvision removed")

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
torchvision removed


In [13]:
# Cell  - Clean layer_stats.py BEFORE importing EasyEdit
from pathlib import Path
import py_compile

p = Path("/kaggle/working/EasyEdit/easyeditor/models/rome/layer_stats.py")
text = p.read_text()

text = text.replace(
    'raw_ds = load_dataset("wikitext", "wikitext-103-raw-v1")[ds_name]\n        )',
    'raw_ds = load_dataset("wikitext", "wikitext-103-raw-v1")[ds_name]'
)

p.write_text(text)

py_compile.compile(str(p), doraise=True)
print("✅ layer_stats.py fixed before EasyEdit import")

✅ layer_stats.py fixed before EasyEdit import


In [14]:
# FORCE patch LLaMA MLP output for ROME/EasyEdit
from pathlib import Path
import py_compile

llama_path = Path("/kaggle/working/pydeps/transformers/models/llama/modeling_llama.py")
text = llama_path.read_text()

old = """        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states"""

new = """        hidden_states = self.post_attention_layernorm(hidden_states)
        mlp_out = self.mlp(hidden_states)

        # ROME/EasyEdit compatibility: unwrap MLP output if hooks return dict/tuple/list
        def _extract_tensor(obj):
            import torch
            if torch.is_tensor(obj):
                return obj

            if isinstance(obj, dict):
                for key in ["hidden_states", "last_hidden_state", "output", "value"]:
                    if key in obj:
                        found = _extract_tensor(obj[key])
                        if found is not None:
                            return found

                for value in obj.values():
                    found = _extract_tensor(value)
                    if found is not None:
                        return found

                return None

            if isinstance(obj, (tuple, list)):
                for value in obj:
                    found = _extract_tensor(value)
                    if found is not None:
                        return found

                return None

            return None

        mlp_out = _extract_tensor(mlp_out)

        if mlp_out is None:
            raise TypeError("ROME patch failed: self.mlp returned no tensor")

        hidden_states = residual + mlp_out"""

if old not in text and "def _extract_tensor(obj):" not in text:
    print("❌ Original MLP block not found.")
    print("This means modeling_llama.py was already changed in another way.")
    print("Restart Kaggle session and run from the top.")
elif "def _extract_tensor(obj):" in text:
    print("✅ MLP patch already exists")
else:
    text = text.replace(old, new)
    llama_path.write_text(text)
    print("✅ MLP patch applied")

py_compile.compile(str(llama_path), doraise=True)
print("✅ modeling_llama.py syntax OK")

✅ MLP patch applied
✅ modeling_llama.py syntax OK


In [15]:
from pathlib import Path

llama_path = Path("/kaggle/working/pydeps/transformers/models/llama/modeling_llama.py")
lines = llama_path.read_text().splitlines()

for i, line in enumerate(lines):
    if "mlp_out = self.mlp(hidden_states)" in line:
        start = max(0, i - 3)
        end = min(len(lines), i + 25)
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")

742:         # Fully Connected
743:         residual = hidden_states
744:         hidden_states = self.post_attention_layernorm(hidden_states)
745:         mlp_out = self.mlp(hidden_states)
746: 
747:         # ROME/EasyEdit compatibility: unwrap MLP output if hooks return dict/tuple/list
748:         def _extract_tensor(obj):
749:             import torch
750:             if torch.is_tensor(obj):
751:                 return obj
752: 
753:             if isinstance(obj, dict):
754:                 for key in ["hidden_states", "last_hidden_state", "output", "value"]:
755:                     if key in obj:
756:                         found = _extract_tensor(obj[key])
757:                         if found is not None:
758:                             return found
759: 
760:                 for value in obj.values():
761:                     found = _extract_tensor(value)
762:                     if found is not None:
763:                         return found
764: 
765:                 r

In [16]:
from pathlib import Path

llama_path = Path("/kaggle/working/pydeps/transformers/models/llama/modeling_llama.py")
lines = llama_path.read_text().splitlines()

for i in range(767, 790):
    print(f"{i+1}: {lines[i]}")

768:                 for value in obj:
769:                     found = _extract_tensor(value)
770:                     if found is not None:
771:                         return found
772: 
773:                 return None
774: 
775:             return None
776: 
777:         mlp_out = _extract_tensor(mlp_out)
778: 
779:         if mlp_out is None:
780:             raise TypeError("ROME patch failed: self.mlp returned no tensor")
781: 
782:         hidden_states = residual + mlp_out
783: 
784:         outputs = (hidden_states,)
785: 
786:         if output_attentions:
787:             outputs += (self_attn_weights,)
788: 
789:         if use_cache:
790:             outputs += (present_key_value,)


In [17]:
# Cell 13 - Import EasyEdit components
import sys

# Re-enforce correct path order
sys.path = [
    "/kaggle/working/pydeps",
    "/kaggle/working/EasyEdit",
    "/usr/lib/python3.12",
    "/usr/lib/python3.12/lib-dynload",
    "/usr/local/lib/python3.12/dist-packages",
]

from easyeditor.editors.editor import BaseEditor
from easyeditor.models.ike.ike_hparams import IKEHyperParams
from easyeditor.models.rome.rome_hparams import ROMEHyperParams
from easyeditor.dataset.zsre import ZsreDataset

print("Targeted imports OK")

2026-05-04 15:36:34.038896: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777908994.235327      98 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777908994.299737      98 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777908994.790146      98 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777908994.790188      98 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777908994.790191      98 computation_placer.cc:177] computation placer alr

Targeted imports OK


In [18]:
# Cell 14 - HF login
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
login(token=hf_token)
print("HF login done")

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    token=hf_token
)
print(tok.pad_token, tok.eos_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login done


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

None <|eot_id|>


In [19]:
# Cell 15 - ROME hparams
from pathlib import Path
import textwrap

ROME_HPARAMS_PATH = Path("/kaggle/working/EasyEdit/hparams/ROME/llama3.2-1b-instruct.yaml")

ROME_HPARAMS_PATH.write_text(textwrap.dedent("""
alg_name: "ROME"
model_name: "meta-llama/Llama-3.2-1B-Instruct"
device: 0
model_parallel: false
fp16: true
max_length: 40
layers: [4]
v_num_grad_steps: 10
v_lr: 1e-1
clamp_norm_factor: 1.25
kl_factor: 0.25
fact_token: "subject_last"
v_loss_layer: 15
v_weight_decay: 0.5
mom2_adjustment: false
mom2_dataset: "wikitext"
mom2_n_samples: 1000
mom2_dtype: "float32"
stats_dir: "./data/stats"
context_template_length_params:
  - [5, 10]
  - [10, 10]
rewrite_module_tmp: "model.layers.{}.mlp.down_proj"
layer_module_tmp: "model.layers.{}"
mlp_module_tmp: "model.layers.{}.mlp"
attn_module_tmp: "model.layers.{}.self_attn"
ln_f_module: "model.norm"
lm_head_module: "lm_head"
""").strip())

print(ROME_HPARAMS_PATH.read_text())

alg_name: "ROME"
model_name: "meta-llama/Llama-3.2-1B-Instruct"
device: 0
model_parallel: false
fp16: true
max_length: 40
layers: [4]
v_num_grad_steps: 10
v_lr: 1e-1
clamp_norm_factor: 1.25
kl_factor: 0.25
fact_token: "subject_last"
v_loss_layer: 15
v_weight_decay: 0.5
mom2_adjustment: false
mom2_dataset: "wikitext"
mom2_n_samples: 1000
mom2_dtype: "float32"
stats_dir: "./data/stats"
context_template_length_params:
  - [5, 10]
  - [10, 10]
rewrite_module_tmp: "model.layers.{}.mlp.down_proj"
layer_module_tmp: "model.layers.{}"
mlp_module_tmp: "model.layers.{}.mlp"
attn_module_tmp: "model.layers.{}.self_attn"
ln_f_module: "model.norm"
lm_head_module: "lm_head"


In [20]:
from pathlib import Path

compute_v_path = Path("/kaggle/working/EasyEdit/easyeditor/models/rome/compute_v.py")
lines = compute_v_path.read_text().split('\n')

# Find edit_output_fn and replace it entirely
start = None
end = None
for i, line in enumerate(lines):
    if 'def edit_output_fn' in line:
        start = i
    if start and i > start and ('# Optimizer' in line or 'opt = torch' in line):
        end = i
        break

print(f"Replacing lines {start}-{end}")

new_func = [
    '    def edit_output_fn(cur_out, cur_layer):',
    '        nonlocal target_init',
    '        if cur_layer == hparams.mlp_module_tmp.format(layer):',
    '            import torch as _t',
    '            # safely unwrap output to a tensor',
    '            if isinstance(cur_out, _t.Tensor):',
    '                _co = cur_out',
    '            elif isinstance(cur_out, (list, tuple)) and len(cur_out) > 0:',
    '                _co = cur_out[0]',
    '            elif isinstance(cur_out, dict) and len(cur_out) > 0:',
    '                _co = next((v for v in cur_out.values() if isinstance(v, _t.Tensor)), None)',
    '                if _co is None: return cur_out',
    '            else:',
    '                return cur_out',
    '            if target_init is None:',
    '                print("Recording initial value of v*")',
    '                target_init = _co[0, lookup_idxs[0]].detach().clone()',
    '            for i, idx in enumerate(lookup_idxs):',
    '                _co[i, idx, :] += delta',
    '        return cur_out',
]

lines = lines[:start] + new_func + lines[end:]
compute_v_path.write_text('\n'.join(lines))
print("Patched compute_v.py")
print("Verifying:")
result = compute_v_path.read_text().split('\n')
for i in range(start, start + len(new_func) + 2):
    print(i, repr(result[i]))

Replacing lines 82-99
Patched compute_v.py
Verifying:
82 '    def edit_output_fn(cur_out, cur_layer):'
83 '        nonlocal target_init'
84 '        if cur_layer == hparams.mlp_module_tmp.format(layer):'
85 '            import torch as _t'
86 '            # safely unwrap output to a tensor'
87 '            if isinstance(cur_out, _t.Tensor):'
88 '                _co = cur_out'
89 '            elif isinstance(cur_out, (list, tuple)) and len(cur_out) > 0:'
90 '                _co = cur_out[0]'
91 '            elif isinstance(cur_out, dict) and len(cur_out) > 0:'
92 '                _co = next((v for v in cur_out.values() if isinstance(v, _t.Tensor)), None)'
93 '                if _co is None: return cur_out'
94 '            else:'
95 '                return cur_out'
96 '            if target_init is None:'
97 '                print("Recording initial value of v*")'
98 '                target_init = _co[0, lookup_idxs[0]].detach().clone()'
99 '            for i, idx in enumerate(lookup_idx

In [21]:
import torch
import gc

# Free the old model from GPU
try:
    del editor
    del edited_model
except NameError:
    pass

torch.cuda.empty_cache()
gc.collect()

print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

GPU memory free: 15.53 GB


In [22]:
# Fix EasyEdit editor.py missing LORA name
import easyeditor.editors.editor as editor_module

if not hasattr(editor_module, "LORA"):
    class LORA:
        pass

    editor_module.LORA = LORA

print("✅ Patched missing LORA name in editor module")

✅ Patched missing LORA name in editor module


In [23]:
# Patch editor.py so missing LORA does not crash ROME
from pathlib import Path
import py_compile

editor_path = Path("/kaggle/working/EasyEdit/easyeditor/editors/editor.py")
text = editor_path.read_text()

old = "        if isinstance(edited_model, LORA):"
new = "        if 'LORA' in globals() and isinstance(edited_model, LORA):"

if old in text:
    text = text.replace(old, new)
    editor_path.write_text(text)
    print("✅ Patched editor.py LORA check")
elif new in text:
    print("✅ editor.py LORA check already patched")
else:
    print("⚠️ LORA check line not found")

py_compile.compile(str(editor_path), doraise=True)
print("✅ editor.py syntax OK")

✅ Patched editor.py LORA check
✅ editor.py syntax OK


In [24]:
# Cell 16 - Create editor and verify nethook only
from easyeditor.editors.editor import BaseEditor
from easyeditor.models.rome.rome_hparams import ROMEHyperParams
from easyeditor.util import nethook
import torch

hparams = ROMEHyperParams.from_hparams(
    "/kaggle/working/EasyEdit/hparams/ROME/llama3.2-1b-instruct.yaml"
)

editor = BaseEditor.from_hparams(hparams)

# Verify nethook
model = editor.model
model.eval()

layer0_mlp = model.model.layers[0].mlp
dtype = next(model.parameters()).dtype
device = next(model.parameters()).device
hidden_size = model.config.hidden_size

x = torch.randn(1, 4, hidden_size, device=device, dtype=dtype)

with torch.no_grad():
    with nethook.Trace(layer0_mlp, retain_output=True) as tr:
        y = layer0_mlp(x)

print("MLP return type:", type(y))
print("MLP return is tensor:", torch.is_tensor(y))
print("Trace output type:", type(tr.output))
print("Trace output is tensor:", torch.is_tensor(tr.output))

if not torch.is_tensor(y):
    raise TypeError(f"nethook is still corrupting MLP output: {type(y)}")

print("✅ editor ready")
print("✅ nethook test passed")

We are creating the logger files


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

MLP return type: <class 'torch.Tensor'>
MLP return is tensor: True
Trace output type: <class 'torch.Tensor'>
Trace output is tensor: True
✅ editor ready
✅ nethook test passed


In [25]:
# Cell 17 - Load KnowEdit ZsRE manually from Hugging Face

from huggingface_hub import hf_hub_download
import json

zsre_path = hf_hub_download(
    repo_id="zjunlp/KnowEdit",
    repo_type="dataset",
    filename="benchmark/ZsRE/ZsRE-test-all.json",
)

with open(zsre_path, "r") as f:
    zsre_rows = json.load(f)

print("ZsRE rows:", len(zsre_rows))
print("\nFirst row:")
print(zsre_rows[0])

ZsRE-test-all.json: 0.00B [00:00, ?B/s]

ZsRE rows: 1301

First row:
{'subject': 'Epaspidoceras', 'target_new': 'Noctuidae', 'prompt': 'Which family does Epaspidoceras belong to?', 'ground_truth': ['Aspidoceratidae'], 'rephrase_prompt': 'What family are Epaspidoceras?', 'cond': 'Geometridae >> Noctuidae || Which family does Epaspidoceras belong to?', 'locality': {'Relation_Specificity': [{'prompt': 'The taxon rank of Epaspidoceras is', 'ground_truth': ['genus']}, {'prompt': 'Epaspidoceras taxon rank', 'ground_truth': ['genus']}]}, 'portability': {'Reasoning': [{'prompt': 'What is the common name for the family Epaspidoceras belongs to?', 'ground_truth': 'Owlet moths'}]}}


In [26]:
# Cell 18 - Convert ZsRE rows to EasyEdit ROME format

N = 100
rows_source = zsre_rows

def clean_answer(x):
    x = str(x).strip()
    if x.endswith("."):
        x = x[:-1].strip()
    return x

def get_rephrase(row):
    # ZsRE may store paraphrases under different names depending on formatting
    for key in ["rephrase", "rephrase_prompt", "rephrase_prompts"]:
        if key in row and row[key]:
            r = row[key]
            if isinstance(r, list):
                return str(r[0]).strip() if len(r) > 0 else None
            return str(r).strip()
    return None

rows = []
prompts = []
target_new = []
ground_truth = []
subject = []
rephrase_prompts = []

bad_rows = []

for i, x in enumerate(rows_source):
    try:
        p = x["prompt"]
        tn = clean_answer(x["target_new"])
        gt = clean_answer(x["ground_truth"])
        s = x["subject"]

        rp = get_rephrase(x)

        # Keep rows even if no rephrase exists.
        # If no rephrase, use the original prompt as fallback so Cell 19 does not crash.
        if rp is None:
            rp = p

        rows.append(x)
        prompts.append(p)
        target_new.append(tn)
        ground_truth.append(gt)
        subject.append(s)
        rephrase_prompts.append(rp)

        if len(rows) >= N:
            break

    except Exception as e:
        bad_rows.append((i, str(e), x))

print("Usable ZsRE samples:", len(prompts))
print("Bad rows:", len(bad_rows))

print("\nExample converted sample:")
print("prompt:", prompts[0])
print("rephrase:", rephrase_prompts[0])
print("target_new:", target_new[0])
print("ground_truth:", ground_truth[0])
print("subject:", subject[0])
print("row keys:", rows[0].keys())

Usable ZsRE samples: 1000
Bad rows: 0

Example converted sample:
prompt: Which family does Epaspidoceras belong to?
rephrase: What family are Epaspidoceras?
target_new: Noctuidae
ground_truth: ['Aspidoceratidae']
subject: Epaspidoceras
row keys: dict_keys(['subject', 'target_new', 'prompt', 'ground_truth', 'rephrase_prompt', 'cond', 'locality', 'portability'])


In [27]:
# Cell 19 - Run ROME with rewrite, paraphrase/rephrase, portability, and locality

import json
import ast
import gc
import torch
from pathlib import Path

# Start small first. Change to 100 after it works.
N = min(100, len(prompts))

OUT_PARTIAL = Path("/kaggle/working/rome_100_with_rephrase_partial.json")
OUT_FINAL = Path("/kaggle/working/rome_100_with_rephrase.json")

all_metrics = []
failed = []

def parse_maybe_json(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        if x.strip() == "":
            return {}
        try:
            return json.loads(x)
        except Exception:
            try:
                return ast.literal_eval(x)
            except Exception:
                return {}
    return {}

def get_text_value(x):
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, dict):
        if "str" in x:
            return str(x["str"]).strip()
        if "text" in x:
            return str(x["text"]).strip()
        if "answer" in x:
            return str(x["answer"]).strip()
    return str(x).strip()

def normalize_eval_inputs(raw_eval):
    """
    Converts KnowEdit locality/portability fields into EasyEdit format.

    EasyEdit expects, for one edit:
    {
        "metric_name": {
            "prompt": ["..."],
            "ground_truth": ["..."]
        }
    }
    """
    raw_eval = parse_maybe_json(raw_eval)

    if raw_eval is None or raw_eval == {} or raw_eval == []:
        return {}

    out = {}

    if isinstance(raw_eval, list):
        for j, item in enumerate(raw_eval):
            if not isinstance(item, dict):
                continue

            p = item.get("prompt") or item.get("question") or item.get("input")
            gt = (
                item.get("ground_truth")
                or item.get("target")
                or item.get("answer")
                or item.get("target_new")
            )

            if p is not None and gt is not None:
                out[f"eval_{j}"] = {
                    "prompt": [get_text_value(p)],
                    "ground_truth": [get_text_value(gt)],
                }

    elif isinstance(raw_eval, dict):
        for key, value in raw_eval.items():
            if isinstance(value, dict):
                p = value.get("prompt") or value.get("question") or value.get("input")
                gt = (
                    value.get("ground_truth")
                    or value.get("target")
                    or value.get("answer")
                    or value.get("target_new")
                )

                if p is not None and gt is not None:
                    out[str(key)] = {
                        "prompt": [get_text_value(p)],
                        "ground_truth": [get_text_value(gt)],
                    }

            elif isinstance(value, list):
                for j, item in enumerate(value):
                    if not isinstance(item, dict):
                        continue

                    p = item.get("prompt") or item.get("question") or item.get("input")
                    gt = (
                        item.get("ground_truth")
                        or item.get("target")
                        or item.get("answer")
                        or item.get("target_new")
                    )

                    if p is not None and gt is not None:
                        out[f"{key}_{j}"] = {
                            "prompt": [get_text_value(p)],
                            "ground_truth": [get_text_value(gt)],
                        }

    return out


for i in range(N):
    try:
        prompt = prompts[i]
        target = target_new[i]
        truth = ground_truth[i]
        subj = subject[i]

        # This is the important fix for paraphrase/rephrase accuracy
        rp = rephrase_prompts[i]

        row = rows[i]

        locality_inputs = normalize_eval_inputs(row.get("locality"))
        portability_inputs = normalize_eval_inputs(row.get("portability"))

        metrics, _, _ = editor.edit(
            prompts=[prompt],
            target_new=[target],
            ground_truth=[truth],
            subject=[subj],
            rephrase_prompts=[rp],
            locality_inputs=locality_inputs if locality_inputs else None,
            portability_inputs=portability_inputs if portability_inputs else None,
            sequential_edit=False,
            keep_original_weight=False,
        )

        all_metrics.extend(metrics)

        print("\n" + "=" * 80)
        print(f"Sample {i}")
        print("Prompt:", prompt)
        print("Rephrase:", rp)
        print("Target:", target)
        print("Subject:", subj)
        print("Metrics Summary:", {
            "pre": metrics[0].get("pre", {}),
            "post": metrics[0].get("post", {}),
        })

    except Exception as e:
        failed.append((i, str(e)))
        print(f"\n❌ Sample {i} failed:", e)

    if (i + 1) % 25 == 0:
        with open(OUT_PARTIAL, "w") as f:
            json.dump(all_metrics, f, indent=2)

        gc.collect()
        torch.cuda.empty_cache()

        print(f"\nFinished {i + 1}/{N} | failed: {len(failed)}")

with open(OUT_FINAL, "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\nDone.")
print("Total metrics:", len(all_metrics))
print("Failed:", len(failed))
print("Saved:", OUT_FINAL)

if failed:
    print("First failures:", failed[:5])

  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [Which family does Epaspidoceras belong to?] -> [ Noctuidae]


We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


Cached context templates ['{}', 'The following is a. {}', 'The 1970. {}', 'Therefore, if you. {}', 'Therefore\n**The. {}', 'Because we care about. {}', 'Because of this,. {}', 'I think there may. {}', 'I am writing a. {}', "You're the star. {}", 'You can find the. {}', 'The 2023 World Cup is just around. {}', 'The first and only "Golden Age" of. {}', 'Therefore I am a part of the group of. {}', 'Therefore, the best answer is (B)(. {}', "Because You're a Good Guy: A Guide. {}", 'Because of its rich cultural heritage, stunning natural. {}', 'I am excited to announce that the new season. {}', "I've been trying to get into a new. {}", 'Youthful vigor is in the blood,. {}', 'Youth in the Digital Age\nYouth. {}']
Computing left vector (u)...
Selected u projection object Epaspidoceras
Left vector shape: torch.Size([8192])
Computing right vector (v)
Lookup index found: 8 | Sentence: Which family does Epaspidoceras belong to? Noctuid | Token: eras
Rewrite layer is 4
Tying optimization objective

100%|██████████| 1/1 [00:02<00:00,  2.14s/it]


...

loss 1.688 = 1.432 + 0.018 + 0.238 avg prob of [ 20 December 2007] 0.24029573798179626
Delta norm: 3.28125
Change in target norm: 2.625 to 4.1796875 => 1.5546875
Division Factor: 3.31640625
Right vector norm: 0.9892578125
Right vector shape: torch.Size([2048])
Deltas successfully computed for ['model.layers.4.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.4.mlp.down_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': 0.16666666666666666, 'rephrase_acc': 0.16666666666666666, 'portability': {'Subject_Aliasing_0_acc': 0.16666666666666666}}, 'post': {'rewrite_acc': 0.6666666666666666, 'rephrase_acc': 0.5, 'locality': {'Relation_Specificity_1_acc': 0.8333333333333334, 'Relation_Specificity_0_acc': 0.8333333333333334}, 'portability': {'Subject_Aliasing_0_acc': 0.16666666666666666}}}

Sample 8
Prompt: What day was USA-199 launched?
Rephrase: When was the launch date for USA-199?
Target: 20 December 2007
Subject: USA-199
Metrics Summary: {'pre': {'rewrite_acc'

  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [What was the record label of Runaway Sunday?] -> [ Motown]
Computing left vector (u)...
Selected u projection object Runaway Sunday
Left vector shape: torch.Size([8192])
Computing right vector (v)
Lookup index found: 9 | Sentence: What was the record label of Runaway Sunday? Mot | Token:  Sunday
Rewrite layer is 4
Tying optimization objective to 15
Recording initial value of v*
loss 4.502 = 4.502 + 0.0 + 0.0 avg prob of [ Motown] 0.011941373348236084
loss 2.396 = 1.949 + 0.225 + 0.222 avg prob of [ Motown] 0.15341436862945557
loss 0.42 = 0.097 + 0.101 + 0.222 avg prob of [ Motown] 0.90836101770401
loss 0.314 = 0.016 + 0.076 + 0.222 avg prob of [ Motown] 0.9842514991760254
loss 0.294 = 0.013 + 0.06 + 0.222 avg prob of [ Motown] 0.9875307083129883
loss 0.283 = 0.012 + 0.048 + 0.222 avg prob of [ Motown] 0.9876390099525452
loss 0.271 = 0.012 + 0.037 + 0.222 avg prob of [ Motown] 0.9885693788528442
loss 0.263 = 0.01 + 0.03 + 0.222 avg prob of [ Mot

100%|██████████| 1/1 [00:01<00:00,  1.45s/it]

loss 0.253 = 0.007 + 0.023 + 0.222 avg prob of [ Motown] 0.9926832318305969
Delta norm: 3.515625
Change in target norm: 2.8125 to 4.48828125 => 1.67578125
Division Factor: 3.76953125
Right vector norm: 0.9326171875
Right vector shape: torch.Size([2048])
Deltas successfully computed for ['model.layers.4.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.4.mlp.down_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': 0.5, 'rephrase_acc': 0.5, 'portability': {'Reasoning_0_acc': 0.6666666666666666}}, 'post': {'rewrite_acc': 1.0, 'rephrase_acc': 1.0, 'locality': {'Relation_Specificity_1_acc': 0.75, 'Relation_Specificity_0_acc': 1.0}, 'portability': {'Reasoning_0_acc': 0.6666666666666666}}}

Sample 9
Prompt: What was the record label of Runaway Sunday?
Rephrase: What was Runaway Sunday's record label?
Target: Motown
Subject: Runaway Sunday
Metrics Summary: {'pre': {'rewrite_acc': [0.5], 'portability': {'Reasoning_0_acc': [0.6666666666666666]}, 'rephrase_acc': [0.

In [28]:
# Cell 20
import pandas as pd
import numpy as np
import json

def flatten_scores(x):
    vals = []

    if isinstance(x, list):
        for item in x:
            vals.extend(flatten_scores(item))

    elif isinstance(x, dict):
        for value in x.values():
            vals.extend(flatten_scores(value))

    elif isinstance(x, (int, float)):
        vals.append(float(x))

    return vals

def avg_score(x):
    vals = flatten_scores(x)
    if len(vals) == 0:
        return None
    return float(np.mean(vals))

table_rows = []

for i, m in enumerate(all_metrics):
    requested = m.get("requested_rewrite", {})
    pre = m.get("pre", {})
    post = m.get("post", {})

    row = {
        "sample_id": i,
        "subject": requested.get("subject"),
        "prompt": requested.get("prompt"),
        "target_new": requested.get("target_new"),
        "ground_truth": requested.get("ground_truth"),

        "pre_rewrite_acc": avg_score(pre.get("rewrite_acc", [])),
        "post_rewrite_acc": avg_score(post.get("rewrite_acc", [])),

        "pre_rephrase_acc": avg_score(pre.get("rephrase_acc", [])),
        "post_rephrase_acc": avg_score(post.get("rephrase_acc", [])),

        "pre_portability_acc": avg_score(pre.get("portability", {})),
        "post_portability_acc": avg_score(post.get("portability", {})),

        "pre_locality_acc": avg_score(pre.get("locality", {})),
        "post_locality_acc": avg_score(post.get("locality", {})),
    }

    table_rows.append(row)

df = pd.DataFrame(table_rows)

display(df.head(20))


,sample_id,subject,prompt,target_new,ground_truth,pre_rewrite_acc,post_rewrite_acc,pre_rephrase_acc,post_rephrase_acc,pre_portability_acc,post_portability_acc,pre_locality_acc,post_locality_acc
0,0,Epaspidoceras,Which family does Epaspidoceras belong to?,Noctuidae,['Aspidoceratidae'],0.500000,1.000000,0.500000,1.000000,0.250000,0.250000,None,0.666667
1,1,ZIC3,What species is ZIC3 specific to?,male,['human'],0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,None,0.750000
2,2,Louise Grandjean,What voice type is Louise Grandjean?,mezzo soprano,['soprano'],0.500000,1.000000,0.250000,0.750000,0.400000,0.400000,None,0.666667
3,3,Wang Jipeng,Who is listed as Wang Jipeng father?,Wang Chonghua,['Wang Yanjun'],0.250000,1.000000,0.250000,1.000000,0.250000,0.500000,None,0.625000
4,4,Charlotte of Schaumburg-Lippe,What was the name of Charlotte of Schaumburg-L...,Charlotte of Bourbon-Parma,['Princess Bathildis of Anhalt-Dessau'],0.666667,1.000000,0.500000,0.833333,0.750000,0.750000,None,0.750000
5,5,Butterfly Cluster,What constellation is home to Butterfly Cluster?,Orion,['Scorpius'],0.000000,1.000000,0.000000,1.000000,0.333333,0.666667,None,0.666667
6,6,Juan María Bordaberry,The father of Juan María Bordaberry is whom?,Gabrielle Bordaberry,['Domingo Bordaberry'],0.600000,1.000000,0.600000,1.000000,0.600000,0.600000,None,0.700000
7,7,Javan surili,What level is Javan surili's iucn conservation...,critically threatened,['endangered species'],0.000000,1.000000,0.000000,0.000000,0.500000,0.500000,None,0.500000
8,8,USA-199,What day was USA-199 launched?,20 December 2007,['20 December 2007'],0.166667,0.666667,0.166667,0.500000,0.166667,0.166667,None,0.833333
9,9,Runaway Sunday,What was the record label of Runaway Sunday?,Motown,['Virgin Records'],0.500000,1.000000,0.500000,1.000000,0.666667,0.666667,None,0.875000


In [29]:
# Calculate average metrics from the results table

import pandas as pd

avg_results = pd.DataFrame({
    "metric": [
        "rewrite_acc",
        "rephrase_acc",
        "portability_acc",
        "locality_acc",
    ],
    "pre_average": [
        df["pre_rewrite_acc"].mean(),
        df["pre_rephrase_acc"].mean(),
        df["pre_portability_acc"].mean(),
        df["pre_locality_acc"].mean(),
    ],
    "post_average": [
        df["post_rewrite_acc"].mean(),
        df["post_rephrase_acc"].mean(),
        df["post_portability_acc"].mean(),
        df["post_locality_acc"].mean(),
    ],
})

avg_results["improvement"] = avg_results["post_average"] - avg_results["pre_average"]

display(avg_results)

,metric,pre_average,post_average,improvement
0,rewrite_acc,0.318333,0.966667,0.648333
1,rephrase_acc,0.276667,0.808333,0.531667
2,portability_acc,0.391667,0.450000,0.058333
3,locality_acc,NaN,0.703333,NaN
